# Home Credit Default Risk — Modeling Pipeline

## Pipeline Overview
```
Step 1: Load preprocessed datasets (4 variants)
         ├── traditional_pca
         ├── traditional_no_pca
         ├── combined_pca
         └── combined_no_pca
Step 2: Baseline Models (LR, LightGBM, XGBoost, RandomForest)
         └── Ensemble downsampling: majority split into K folds,
             each fold trains all minority + 1/K majority → average probas
Step 3: Evaluation Tables + Plots per dataset variant
         ├── Metrics table (Recall, Precision, F1, Accuracy, AUC)
         ├── AUC Bar Chart across all variants
         ├── ROC Curves
         ├── Precision-Recall Curves
         ├── Confusion Matrices
         ├── Feature Importances (tree models)
         ├── Lift Chart + KS Plot
         └── Cross-Dataset AUC Heatmap
Step 4: Hyperparameter Tuning (RandomizedSearchCV, n_iter=5)
         ├── Best params log
         ├── Baseline vs Tuned AUC comparison
         └── Tuned ROC + AUC heatmap
Step 5: Final evaluation on test set with best tuned models
         ├── Test metrics tables
         ├── Test ROC Curves
         ├── Final AUC heatmap
         └── Overall best model summary
Step 6: SHAP Analysis
         ├── Beeswarm summary plot
         ├── Bar importance plot
         ├── SHAP vs Native importance comparison
         ├── Dependence plots (top 4 features)
         ├── Waterfall plots (top 3 high-risk samples)
         ├── SHAP heatmap
         └── SHAP feature importance table
```

## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')
shap.initjs()

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score, accuracy_score, roc_auc_score,
    roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils import shuffle
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import pipelines.project_config as cfg

sns.set_style('whitegrid')
SEED = 42
DO_HYPERTUNING      = False
N_ENSEMBLE_FOLDS    = 3
pd.set_option('display.max_columns', 80)

print('Libraries loaded.')
print(f'Hyperparameter tuning  : {"ENABLED" if DO_HYPERTUNING else "DISABLED"}')
print(f'Ensemble folds (K)     : {N_ENSEMBLE_FOLDS}')
print('Downsampling strategy  : majority split into K folds; each run uses all minority + 1/K majority; probas averaged.')

## 2. Load Datasets

In [ ]:
def load_split(directory):
    train = pd.read_csv(f"{directory}/{cfg.PREPROCESSED_FILENAMES['train']}")
    val   = pd.read_csv(f"{directory}/{cfg.PREPROCESSED_FILENAMES['validation']}")
    test  = pd.read_csv(f"{directory}/{cfg.PREPROCESSED_FILENAMES['test']}")
    return train, val, test

def xy(df):
    return df.drop(columns=[cfg.TARGET_COL]), df[cfg.TARGET_COL]

datasets = {
    'traditional_pca':    load_split(cfg.PREPROCESSED_TRADITIONAL_PCA_DIR),
    'traditional_no_pca': load_split(cfg.PREPROCESSED_TRADITIONAL_NO_PCA_DIR),
    'combined_pca':       load_split(cfg.PREPROCESSED_COMBINED_PCA_DIR),
    'combined_no_pca':    load_split(cfg.PREPROCESSED_COMBINED_NO_PCA_DIR),
}

print('Dataset shapes:')
for name, (tr, va, te) in datasets.items():
    print(f'  {name:25s}  train={tr.shape}  val={va.shape}  test={te.shape}')

In [ ]:
print('\nChecking for non-numeric columns:')
for dataset_name, (train_df, val_df, test_df) in datasets.items():
    X_train, y_train = xy(train_df)
    object_cols = X_train.select_dtypes(include='object').columns.tolist()
    if object_cols:
        print(f'\n{dataset_name}:')
        print(f'  Found {len(object_cols)} object columns: {object_cols[:5]}...')
    else:
        print(f'\n{dataset_name}: All columns numeric ✓')

## 3. Helper Functions

In [ ]:
def make_ensemble_splits(X, y, K=N_ENSEMBLE_FOLDS, seed=SEED):
    """
    Split majority class into K non-overlapping chunks.
    Each fold contains ALL minority samples + 1/K of the majority samples.
    Returns a list of K (X_fold, y_fold) tuples.
    """
    rng = np.random.default_rng(seed)

    minority_mask = (y == 1)
    X_min = X[minority_mask]
    y_min = y[minority_mask]
    X_maj = X[~minority_mask]
    y_maj = y[~minority_mask]

    maj_idx = rng.permutation(len(X_maj))
    chunks  = np.array_split(maj_idx, K)

    folds = []
    for chunk in chunks:
        X_fold = pd.concat([X_min, X_maj.iloc[chunk]], ignore_index=True)
        y_fold = pd.concat([y_min, y_maj.iloc[chunk]], ignore_index=True)
        X_fold, y_fold = shuffle(X_fold, y_fold, random_state=int(rng.integers(0, 9999)))
        folds.append((X_fold, y_fold))

    minority_pct = len(y_min) / (len(y_min) + len(maj_idx) // K) * 100
    print(f'  Ensemble folds: {K}  |  minority samples: {len(y_min)}  '
          f'|  majority per fold: ~{len(maj_idx)//K}  '
          f'|  minority % per fold: ~{minority_pct:.1f}%')
    return folds


def train_ensemble(base_model_fn, X, y, K=N_ENSEMBLE_FOLDS, seed=SEED):
    """
    Train K models, one per ensemble fold.
    base_model_fn() must return a fresh, unfitted model.
    Returns a list of K fitted models.
    """
    folds  = make_ensemble_splits(X, y, K=K, seed=seed)
    models = []
    for i, (X_fold, y_fold) in enumerate(folds):
        m = base_model_fn()
        m.fit(X_fold, y_fold)
        models.append(m)
    return models


def predict_proba_ensemble(models, X):
    """Average predicted probabilities across all K models."""
    probas = np.stack([m.predict_proba(X)[:, 1] for m in models], axis=0)
    return probas.mean(axis=0)


def evaluate_ensemble(models, X, y, threshold=0.5):
    proba = predict_proba_ensemble(models, X)
    preds = (proba >= threshold).astype(int)
    return {
        'Recall':    recall_score(y, preds, zero_division=0),
        'Precision': precision_score(y, preds, zero_division=0),
        'F1':        f1_score(y, preds, zero_division=0),
        'Accuracy':  accuracy_score(y, preds),
        'AUC':       roc_auc_score(y, proba),
    }


def make_model_factory(scale_pos_weight=1.0):
    """Returns a dict of {model_name: callable} where each call gives a fresh model."""
    return {
        'LogisticRegression': lambda: LogisticRegression(
            max_iter=3000, random_state=SEED, class_weight='balanced',
            solver='saga', penalty='l2', C=0.1
        ),
        'LightGBM': lambda: LGBMClassifier(
            n_estimators=1000, learning_rate=0.05, num_leaves=64,
            max_depth=8, class_weight='balanced', subsample=0.8,
            colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
            random_state=SEED, verbose=-1, importance_type='gain'
        ),
        'XGBoost': lambda: XGBClassifier(
            n_estimators=1000, learning_rate=0.05, max_depth=8,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight, tree_method='hist',
            random_state=SEED, eval_metric='logloss'
        ),
        'RandomForest': lambda: RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_leaf=20,
            max_features='sqrt', class_weight='balanced_subsample',
            random_state=SEED
        ),
    }

MODEL_COLORS = {
    'LogisticRegression': '#1976d2',
    'LightGBM':           '#4caf50',
    'XGBoost':            '#ff9800',
    'RandomForest':       '#9c27b0',
}

## 4. Baseline Models — Train & Evaluate

> **Ensemble downsampling**: the majority class is split into **K = `N_ENSEMBLE_FOLDS`** non-overlapping chunks. For each chunk we train a fresh model on *all* minority samples + that 1/K slice of the majority. Final predicted probabilities are the mean across all K models.

In [ ]:
baseline_fitted  = {}
baseline_results = {}
baseline_probas  = {}

for dataset_name, (train_df, val_df, test_df) in datasets.items():
    print(f'\n[{dataset_name}]')
    X_train, y_train = xy(train_df)
    X_val,   y_val   = xy(val_df)

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    factories = make_model_factory(scale_pos_weight=scale_pos_weight)

    rows   = {}
    probas = {}
    fitted = {}

    for model_name, model_fn in factories.items():
        print(f'  Training {model_name} ({N_ENSEMBLE_FOLDS} folds)...')
        ensemble = train_ensemble(model_fn, X_train, y_train, K=N_ENSEMBLE_FOLDS)
        fitted[model_name]  = ensemble
        rows[model_name]    = evaluate_ensemble(ensemble, X_val, y_val)
        probas[model_name]  = predict_proba_ensemble(ensemble, X_val)
        print(f'    → AUC={rows[model_name]["AUC"]:.4f}  Recall={rows[model_name]["Recall"]:.4f}')

    baseline_fitted[dataset_name]  = fitted
    baseline_results[dataset_name] = rows
    baseline_probas[dataset_name]  = (probas, y_val)

print('\nBaseline training complete.')

### 4.1 Baseline Metrics Tables

In [ ]:
for dataset_name, rows in baseline_results.items():
    print(f'\n=== {dataset_name} ===')
    df_res = pd.DataFrame(rows).T[['Recall', 'Precision', 'F1', 'Accuracy', 'AUC']]
    display(df_res.style.format('{:.4f}').background_gradient(cmap='Blues', axis=0).set_caption(dataset_name))

### 4.2 Baseline AUC Bar Chart — All Dataset Variants

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=False)

for ax, (dataset_name, rows) in zip(axes, baseline_results.items()):
    model_names = list(rows.keys())
    aucs = [rows[m]['AUC'] for m in model_names]
    colors = [MODEL_COLORS[m] for m in model_names]
    bars = ax.bar(model_names, aucs, color=colors, edgecolor='black', alpha=0.85)
    ax.set_ylim(min(aucs) * 0.97, 1.0)
    ax.set_title(dataset_name.replace('_', '\n'), fontweight='bold', fontsize=10)
    ax.set_ylabel('AUC-ROC')
    ax.tick_params(axis='x', rotation=30)
    for bar, v in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.4f}',
                ha='center', fontsize=8, fontweight='bold')

legend_patches = [mpatches.Patch(color=c, label=m) for m, c in MODEL_COLORS.items()]
fig.legend(handles=legend_patches, loc='upper center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, 1.04))
plt.suptitle('Baseline AUC-ROC by Dataset Variant', fontweight='bold', fontsize=13, y=1.10)
plt.tight_layout()
plt.show()

### 4.3 Cross-Dataset AUC Heatmap

In [ ]:
auc_matrix = pd.DataFrame(
    {ds: {m: baseline_results[ds][m]['AUC'] for m in baseline_results[ds]}
     for ds in baseline_results}
).T

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    auc_matrix.astype(float), annot=True, fmt='.4f',
    cmap='YlOrRd', ax=ax, linewidths=0.5,
    vmin=auc_matrix.values.min() * 0.99,
    vmax=auc_matrix.values.max() * 1.005
)
ax.set_title('Baseline AUC-ROC Heatmap — All Models x All Dataset Variants', fontweight='bold', fontsize=13)
ax.set_xlabel('Model')
ax.set_ylabel('Dataset Variant')
plt.tight_layout()
plt.show()

### 4.4 ROC Curves — All Datasets

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, (dataset_name, (probas, y_val)) in zip(axes.flatten(), baseline_probas.items()):
    for model_name, proba in probas.items():
        fpr, tpr, _ = roc_curve(y_val, proba)
        auc = roc_auc_score(y_val, proba)
        ax.plot(fpr, tpr, color=MODEL_COLORS[model_name], linewidth=1.8,
                label=f'{model_name} (AUC={auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves — {dataset_name}', fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('ROC Curves — Baseline Models', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4.5 Precision-Recall Curves — All Datasets

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, (dataset_name, (probas, y_val)) in zip(axes.flatten(), baseline_probas.items()):
    baseline_rate = y_val.mean()
    for model_name, proba in probas.items():
        prec, rec, _ = precision_recall_curve(y_val, proba)
        ax.plot(rec, prec, color=MODEL_COLORS[model_name], linewidth=1.8, label=model_name)
    ax.axhline(y=baseline_rate, color='grey', linestyle='--', alpha=0.6,
               label=f'Baseline ({baseline_rate:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'PR Curves — {dataset_name}', fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Precision-Recall Curves — Baseline Models', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4.6 Confusion Matrices — Best Performing Dataset

In [ ]:
best_dataset = max(
    baseline_results,
    key=lambda d: max(v['AUC'] for v in baseline_results[d].values())
)
_, val_df_best, _ = datasets[best_dataset]
X_val_best, y_val_best = xy(val_df_best)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (model_name, ensemble) in zip(axes, baseline_fitted[best_dataset].items()):
    proba = predict_proba_ensemble(ensemble, X_val_best)
    preds = (proba >= 0.5).astype(int)
    cm = confusion_matrix(y_val_best, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Default', 'Default'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    auc = baseline_results[best_dataset][model_name]['AUC']
    ax.set_title(f'{model_name}\nAUC={auc:.4f}', fontweight='bold', fontsize=10)

plt.suptitle(f'Confusion Matrices — {best_dataset}', fontweight='bold', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

### 4.7 Feature Importances (Tree Models) — Best Dataset

In [ ]:
train_df_best, _, _ = datasets[best_dataset]
X_train_best, _ = xy(train_df_best)
feature_names = X_train_best.columns.tolist()

tree_model_names = ['LightGBM', 'XGBoost', 'RandomForest']

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
for ax, model_name in zip(axes, tree_model_names):
    ensemble = baseline_fitted[best_dataset][model_name]
    avg_importances = np.mean(
        [m.feature_importances_ for m in ensemble], axis=0
    )
    importances = pd.Series(avg_importances, index=feature_names)
    top = importances.nlargest(20).sort_values()
    ax.barh(range(len(top)), top.values,
            color=MODEL_COLORS[model_name], edgecolor='white', alpha=0.85)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([n[:35] for n in top.index], fontsize=8)
    ax.set_title(f'{model_name} — Top 20 Features\n(avg over {N_ENSEMBLE_FOLDS} folds)', fontweight='bold', fontsize=11)
    ax.set_xlabel('Importance')

plt.suptitle(f'Feature Importances — {best_dataset}', fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 4.8 Lift Chart + KS Plot — Best Baseline Model

In [ ]:
best_model_name = max(
    baseline_results[best_dataset],
    key=lambda m: baseline_results[best_dataset][m]['AUC']
)
best_ensemble = baseline_fitted[best_dataset][best_model_name]
best_probas_dict, y_val_best = baseline_probas[best_dataset]
y_proba_best = best_probas_dict[best_model_name]

pred_df = pd.DataFrame({'pred': y_proba_best, 'actual': y_val_best.values})
pred_df['decile'] = pd.qcut(pred_df['pred'], q=10, labels=False, duplicates='drop')
lift = pred_df.groupby('decile')['actual'].mean() / y_val_best.mean()

fpr_k, tpr_k, thresholds_k = roc_curve(y_val_best, y_proba_best)
ks_stat = max(tpr_k - fpr_k)
ks_threshold = thresholds_k[np.argmax(tpr_k - fpr_k)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(
    range(len(lift)), lift.values,
    color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(lift))), edgecolor='black'
)
axes[0].axhline(1.0, color='grey', linestyle='--', alpha=0.7, label='Baseline (1.0x)')
for i, v in enumerate(lift.values):
    axes[0].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontsize=9, fontweight='bold')
axes[0].set_xlabel('Risk Decile (0 = lowest risk, 9 = highest risk)')
axes[0].set_ylabel('Lift vs Baseline')
axes[0].set_title(
    f'Lift Chart — {best_model_name} on {best_dataset}\n(AUC={baseline_results[best_dataset][best_model_name]["AUC"]:.4f})',
    fontweight='bold'
)
axes[0].legend()

axes[1].plot(thresholds_k, tpr_k, 'b-', linewidth=1.8, label='TPR (Sensitivity)')
axes[1].plot(thresholds_k, fpr_k, 'r-', linewidth=1.8, label='FPR (1-Specificity)')
axes[1].fill_between(thresholds_k, tpr_k, fpr_k, alpha=0.08, color='green')
axes[1].axvline(x=ks_threshold, color='green', linestyle='--',
                label=f'KS={ks_stat:.3f} @ {ks_threshold:.3f}')
axes[1].set_xlabel('Threshold')
axes[1].set_title(f'KS Plot (KS={ks_stat:.4f})', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()
print(f'KS Statistic: {ks_stat:.4f}  (KS > 0.3 = production-ready scorecard)')

## 5. Hyperparameter Tuning (RandomizedSearchCV, n_iter=5)

> After tuning, each best estimator is used as the `model_fn` inside the same ensemble downsampling scheme.

In [ ]:
param_grids = {
    'LogisticRegression': {
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear'],
    },
    'LightGBM': {
        'num_leaves': [15, 31, 63],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [100, 200, 500],
        'min_child_samples': [20, 50, 100],
        'pos_weight': [1, 3, 5],
    },
    'XGBoost': {
        'max_depth': [3, 4, 6],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [200, 400],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.6, 0.8],
        'gamma': [0, 1, 5],
    },
    'RandomForest': {
        'n_estimators': [100, 300],
        'max_depth': [10, 20, 30, None],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', None],
        'bootstrap': [True, False]
    },
}

def make_base_estimators_for_tuning(scale_pos_weight=1.0):
    return {
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced'),
        'LightGBM':           LGBMClassifier(random_state=SEED, verbose=-1, class_weight='balanced'),
        'XGBoost':            XGBClassifier(random_state=SEED, eval_metric='logloss', verbosity=0, scale_pos_weight=scale_pos_weight),
        'RandomForest':       RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight='balanced'),
    }

In [ ]:
if DO_HYPERTUNING:
    tuned_models    = {}
    tuned_results   = {}
    tuned_probas    = {}
    best_params_log = {}

    for dataset_name, (train_df, val_df, test_df) in datasets.items():
        print(f'\n[{dataset_name}]')
        X_train, y_train = xy(train_df)
        X_val,   y_val   = xy(val_df)

        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
        base_estimators  = make_base_estimators_for_tuning(scale_pos_weight=scale_pos_weight)

        tuned_models[dataset_name]    = {}
        best_params_log[dataset_name] = {}
        rows   = {}
        probas = {}

        first_fold_X, first_fold_y = make_ensemble_splits(X_train, y_train, K=N_ENSEMBLE_FOLDS)[0]

        for model_name, estimator in base_estimators.items():
            search = RandomizedSearchCV(
                estimator=estimator,
                param_distributions=param_grids[model_name],
                n_iter=10, scoring='recall', cv=3,
                random_state=SEED, n_jobs=1,
            )
            search.fit(first_fold_X, first_fold_y)
            best_params = search.best_params_
            best_params_log[dataset_name][model_name] = best_params

            import copy
            def make_tuned_fn(est, params):
                def _fn():
                    m = copy.deepcopy(est)
                    m.set_params(**params)
                    return m
                return _fn

            ensemble = train_ensemble(
                make_tuned_fn(estimator, best_params),
                X_train, y_train, K=N_ENSEMBLE_FOLDS
            )
            tuned_models[dataset_name][model_name] = ensemble
            rows[model_name]   = evaluate_ensemble(ensemble, X_val, y_val)
            probas[model_name] = predict_proba_ensemble(ensemble, X_val)
            print(f'  {model_name:20s}  Recall={rows[model_name]["Recall"]:.4f}  AUC={rows[model_name]["AUC"]:.4f}')

        tuned_results[dataset_name] = rows
        tuned_probas[dataset_name]  = (probas, y_val)

    print('\nTuning complete.')
else:
    print('Hyperparameter tuning DISABLED. Using baseline models for final evaluation.')
    tuned_models    = None
    tuned_results   = None
    tuned_probas    = None
    best_params_log = None

### 5.1 Best Hyperparameters Found

In [ ]:
if DO_HYPERTUNING:
    for dataset_name, params_by_model in best_params_log.items():
        print(f'\n=== {dataset_name} ===')
        for model_name, params in params_by_model.items():
            print(f'  {model_name:20s}: {params}')
else:
    print('Hyperparameter tuning is disabled. Skipping best parameters display.')

### 5.2 Tuned Metrics Tables

In [ ]:
if DO_HYPERTUNING:
    for dataset_name, rows in tuned_results.items():
        print(f'\n=== {dataset_name} (tuned) ===')
        df_res = pd.DataFrame(rows).T[['Recall', 'Precision', 'F1', 'Accuracy', 'AUC']]
        display(df_res.style.format('{:.4f}').background_gradient(cmap='Greens', axis=0).set_caption(f'{dataset_name} — tuned'))
else:
    print('Hyperparameter tuning is disabled. Showing baseline metrics instead.')
    for dataset_name, rows in baseline_results.items():
        print(f'\n=== {dataset_name} (baseline) ===')
        df_res = pd.DataFrame(rows).T[['Recall', 'Precision', 'F1', 'Accuracy', 'AUC']]
        display(df_res.style.format('{:.4f}').background_gradient(cmap='Blues', axis=0).set_caption(f'{dataset_name} — baseline (used for testing)'))

### 5.3 Baseline vs Tuned AUC — All Datasets

In [ ]:
if DO_HYPERTUNING:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    for ax, dataset_name in zip(axes.flatten(), datasets.keys()):
        model_names = list(baseline_results[dataset_name].keys())
        base_aucs   = [baseline_results[dataset_name][m]['AUC']  for m in model_names]
        tuned_aucs  = [tuned_results[dataset_name][m]['AUC'] for m in model_names]

        x = np.arange(len(model_names))
        width = 0.35
        bars1 = ax.bar(x - width/2, base_aucs,  width, label='Baseline', color='#90caf9', edgecolor='black')
        bars2 = ax.bar(x + width/2, tuned_aucs, width, label='Tuned',    color='#1976d2', edgecolor='black')

        ax.set_xticks(x)
        ax.set_xticklabels(model_names, rotation=15, fontsize=9)
        ax.set_ylabel('AUC-ROC')
        ax.set_title(dataset_name, fontweight='bold', fontsize=11)
        ax.legend(fontsize=9)
        ax.set_ylim(min(base_aucs + tuned_aucs) * 0.97, 1.0)

        for bar, v in zip(bars1, base_aucs):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.4f}', ha='center', fontsize=7)
        for bar, v in zip(bars2, tuned_aucs):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.4f}', ha='center', fontsize=7, fontweight='bold')

    plt.suptitle('Baseline vs Tuned AUC — All Datasets', fontweight='bold', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Hyperparameter tuning is disabled. Skipping comparison chart.')

### 5.4 ROC Curves — Tuned Models

In [ ]:
if DO_HYPERTUNING:
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))

    for ax, (dataset_name, (probas, y_val)) in zip(axes.flatten(), tuned_probas.items()):
        for model_name, proba in probas.items():
            fpr, tpr, _ = roc_curve(y_val, proba)
            auc = roc_auc_score(y_val, proba)
            ax.plot(fpr, tpr, color=MODEL_COLORS[model_name], linewidth=1.8,
                    label=f'{model_name} (AUC={auc:.4f})')
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC — {dataset_name} (tuned)', fontweight='bold', fontsize=11)
        ax.legend(fontsize=8, loc='lower right')
        ax.grid(True, alpha=0.3)

    plt.suptitle('ROC Curves — Tuned Models', fontweight='bold', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Hyperparameter tuning is disabled. Skipping tuned ROC curves.')

### 5.5 Tuned AUC Heatmap

In [ ]:
if DO_HYPERTUNING:
    tuned_auc_matrix = pd.DataFrame(
        {ds: {m: tuned_results[ds][m]['AUC'] for m in tuned_results[ds]}
         for ds in tuned_results}
    ).T

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        tuned_auc_matrix.astype(float), annot=True, fmt='.4f',
        cmap='YlOrRd', ax=ax, linewidths=0.5,
        vmin=tuned_auc_matrix.values.min() * 0.99,
        vmax=tuned_auc_matrix.values.max() * 1.005
    )
    ax.set_title('Tuned AUC-ROC Heatmap — All Models x All Dataset Variants', fontweight='bold', fontsize=13)
    ax.set_xlabel('Model')
    ax.set_ylabel('Dataset Variant')
    plt.tight_layout()
    plt.show()
else:
    print('Hyperparameter tuning is disabled. Skipping tuned AUC heatmap.')

## 6. Final Evaluation on Test Set (Best Tuned Models)

In [ ]:
test_results = {}
test_probas  = {}

models_to_use = tuned_models if DO_HYPERTUNING else baseline_fitted
model_source  = 'Tuned' if DO_HYPERTUNING else 'Baseline'

for dataset_name, (train_df, val_df, test_df) in datasets.items():
    X_test, y_test = xy(test_df)
    rows   = {}
    probas = {}
    for model_name, ensemble in models_to_use[dataset_name].items():
        rows[model_name]   = evaluate_ensemble(ensemble, X_test, y_test)
        probas[model_name] = predict_proba_ensemble(ensemble, X_test)
    test_results[dataset_name] = rows
    test_probas[dataset_name]  = (probas, y_test)

print(f'Test evaluation complete using {model_source} models.')

### 6.1 Test Set Metrics Tables

In [ ]:
model_source = 'Tuned' if DO_HYPERTUNING else 'Baseline'
for dataset_name, rows in test_results.items():
    print(f'\n=== {dataset_name} — Test Set ({model_source} Models) ===')
    df_res = pd.DataFrame(rows).T[['Recall', 'Precision', 'F1', 'Accuracy', 'AUC']]
    display(df_res.style.format('{:.4f}').background_gradient(cmap='Oranges', axis=0).set_caption(f'{dataset_name} — test ({model_source})'))

### 6.2 Test ROC Curves

In [ ]:
model_source = 'Tuned' if DO_HYPERTUNING else 'Baseline'
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, (dataset_name, (probas, y_test)) in zip(axes.flatten(), test_probas.items()):
    for model_name, proba in probas.items():
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax.plot(fpr, tpr, color=MODEL_COLORS[model_name], linewidth=1.8,
                label=f'{model_name} (AUC={auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Test ROC — {dataset_name}', fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'ROC Curves — {model_source} Models on Test Set', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 6.3 Final Test AUC Heatmap

In [ ]:
model_source = 'Tuned' if DO_HYPERTUNING else 'Baseline'
test_auc_matrix = pd.DataFrame(
    {ds: {m: test_results[ds][m]['AUC'] for m in test_results[ds]}
     for ds in test_results}
).T

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    test_auc_matrix.astype(float), annot=True, fmt='.4f',
    cmap='YlOrRd', ax=ax, linewidths=0.5,
    vmin=test_auc_matrix.values.min() * 0.99,
    vmax=test_auc_matrix.values.max() * 1.005
)
ax.set_title(f'Final Test AUC-ROC Heatmap ({model_source} Models) — All Models x All Dataset Variants', fontweight='bold', fontsize=13)
ax.set_xlabel('Model')
ax.set_ylabel('Dataset Variant')
plt.tight_layout()
plt.show()

### 6.4 Overall Best Model Summary

In [ ]:
model_source = 'Tuned' if DO_HYPERTUNING else 'Baseline'
summary_rows = []
for ds, rows in test_results.items():
    for model_name, metrics in rows.items():
        summary_rows.append({'Dataset': ds, 'Model': model_name, **metrics})

summary_df = pd.DataFrame(summary_rows).sort_values('AUC', ascending=False).reset_index(drop=True)
summary_df[['Recall', 'Precision', 'F1', 'Accuracy', 'AUC']] =     summary_df[['Recall', 'Precision', 'F1', 'Accuracy', 'AUC']].round(4)

print(f'=== TOP 10 MODEL-DATASET COMBINATIONS BY TEST AUC ({model_source} Models) ===')
display(summary_df.head(10).style.background_gradient(subset=['AUC'], cmap='YlOrRd'))

best_row = summary_df.iloc[0]
print(f'\nOverall Best: {best_row["Model"]} on {best_row["Dataset"]}')
print(f'  AUC={best_row["AUC"]:.4f}  F1={best_row["F1"]:.4f}  Recall={best_row["Recall"]:.4f}  Precision={best_row["Precision"]:.4f}')
print(f'  Models used: {model_source}')

## 7. SHAP Analysis — Best Model

> SHAP is run on the **first fold model** of the best ensemble (fold 0).  
> All K fold-models share the same feature space; fold 0 is representative.

In [ ]:
shap_dataset    = best_dataset
shap_model_name = best_model_name
shap_ensemble   = baseline_fitted[shap_dataset][shap_model_name]
shap_model      = shap_ensemble[0]

train_df_shap, val_df_shap, _ = datasets[shap_dataset]
X_train_shap, y_train_shap = xy(train_df_shap)
X_val_shap,   y_val_shap   = xy(val_df_shap)

N_SHAP_BACKGROUND = 500
N_SHAP_EXPLAIN    = 1000

rng    = np.random.default_rng(SEED)
bg_idx  = rng.choice(len(X_train_shap), size=min(N_SHAP_BACKGROUND, len(X_train_shap)), replace=False)
exp_idx = rng.choice(len(X_val_shap),   size=min(N_SHAP_EXPLAIN,    len(X_val_shap)),   replace=False)

X_bg  = X_train_shap.iloc[bg_idx]
X_exp = X_val_shap.iloc[exp_idx]

print(f'Running SHAP on: {shap_model_name} (fold 0) | {shap_dataset}')

if shap_model_name in ('LightGBM', 'XGBoost', 'RandomForest'):
    explainer   = shap.TreeExplainer(shap_model, data=X_bg, feature_perturbation='interventional', model_output='probability')
    shap_values = explainer(X_exp)
else:
    explainer = shap.KernelExplainer(shap_model.predict_proba, shap.sample(X_bg, 100))
    sv_raw    = explainer.shap_values(X_exp, nsamples=200)
    shap_values = shap.Explanation(
        values=sv_raw[1],
        base_values=np.full(len(X_exp), explainer.expected_value[1]),
        data=X_exp.values,
        feature_names=X_exp.columns.tolist()
    )

print(f'SHAP values computed for {len(X_exp)} samples across {X_exp.shape[1]} features.')

### 7.1 SHAP Summary Plot (Beeswarm)

In [ ]:
plt.figure(figsize=(10, 9))
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title(f'SHAP Beeswarm — {shap_model_name} on {shap_dataset}', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 7.2 SHAP Feature Importance Bar Plot

In [ ]:
plt.figure(figsize=(9, 8))
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title(f'SHAP Mean |SHAP| Feature Importance — {shap_model_name}', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 7.3 SHAP vs Native Feature Importance (Side-by-Side)

In [ ]:
if shap_model_name in ('LightGBM', 'XGBoost', 'RandomForest'):
    shap_imp = pd.Series(
        np.abs(shap_values.values).mean(axis=0),
        index=X_exp.columns
    ).sort_values(ascending=False).head(20)

    avg_native = np.mean([m.feature_importances_ for m in shap_ensemble], axis=0)
    native_imp = pd.Series(avg_native, index=X_train_shap.columns).reindex(shap_imp.index)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    shap_imp_sorted = shap_imp.sort_values()
    axes[0].barh(range(len(shap_imp_sorted)), shap_imp_sorted.values,
                 color=MODEL_COLORS[shap_model_name], edgecolor='white', alpha=0.85)
    axes[0].set_yticks(range(len(shap_imp_sorted)))
    axes[0].set_yticklabels([n[:35] for n in shap_imp_sorted.index], fontsize=8)
    axes[0].set_title('SHAP Mean |SHAP| Importance', fontweight='bold')
    axes[0].set_xlabel('Mean |SHAP value|')

    native_sorted = native_imp[shap_imp_sorted.index]
    native_norm   = native_sorted / native_sorted.max()
    axes[1].barh(range(len(native_norm)), native_norm.values,
                 color='#78909c', edgecolor='white', alpha=0.85)
    axes[1].set_yticks(range(len(native_norm)))
    axes[1].set_yticklabels([n[:35] for n in native_norm.index], fontsize=8)
    axes[1].set_title(f'Native Feature Importance (avg over {N_ENSEMBLE_FOLDS} folds, normalised)', fontweight='bold')
    axes[1].set_xlabel('Normalised importance')

    plt.suptitle(f'SHAP vs Native Importance — {shap_model_name} | {shap_dataset}',
                 fontweight='bold', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Native importance comparison only available for tree-based models.')

### 7.4 SHAP Dependence Plots — Top 4 Features

In [ ]:
top4_features = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=X_exp.columns
).nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, feat in zip(axes.flatten(), top4_features):
    feat_idx = X_exp.columns.tolist().index(feat)
    shap.dependence_plot(
        feat_idx, shap_values.values, X_exp,
        interaction_index='auto', ax=ax, show=False,
        dot_size=12, alpha=0.5
    )
    ax.set_title(f'SHAP Dependence: {feat[:40]}', fontweight='bold', fontsize=10)

plt.suptitle(f'SHAP Dependence Plots — Top 4 Features | {shap_model_name}',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 7.5 SHAP Waterfall — Top 3 High-Risk Predictions

In [ ]:
proba_exp    = shap_model.predict_proba(X_exp)[:, 1]
top_risk_idx = np.argsort(proba_exp)[-3:][::-1]

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for plot_i, sample_i in enumerate(top_risk_idx):
    plt.sca(axes[plot_i])
    shap.plots.waterfall(shap_values[sample_i], max_display=12, show=False)
    axes[plot_i].set_title(
        f'Sample #{sample_i} | P(default)={proba_exp[sample_i]:.3f}',
        fontweight='bold', fontsize=10
    )

plt.suptitle(f'SHAP Waterfall — Top 3 Highest-Risk Predictions | {shap_model_name}',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 7.6 SHAP Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
shap.plots.heatmap(shap_values, max_display=15, show=False, instance_order=shap_values.sum(1))
plt.title(f'SHAP Heatmap — {shap_model_name} | {shap_dataset}', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 7.7 SHAP Feature Importance Table

In [ ]:
shap_importance_df = pd.DataFrame({
    'Feature':       X_exp.columns,
    'Mean_SHAP':     np.abs(shap_values.values).mean(axis=0),
    'Mean_SHAP_pos': shap_values.values.clip(min=0).mean(axis=0),
    'Mean_SHAP_neg': shap_values.values.clip(max=0).mean(axis=0),
    'Std_SHAP':      np.abs(shap_values.values).std(axis=0),
}).sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)

shap_importance_df[['Mean_SHAP', 'Mean_SHAP_pos', 'Mean_SHAP_neg', 'Std_SHAP']] =     shap_importance_df[['Mean_SHAP', 'Mean_SHAP_pos', 'Mean_SHAP_neg', 'Std_SHAP']].round(6)

print(f'=== SHAP Feature Importance Table — {shap_model_name} | {shap_dataset} ===')
display(
    shap_importance_df.head(30).style
    .background_gradient(subset=['Mean_SHAP'],     cmap='YlOrRd')
    .background_gradient(subset=['Mean_SHAP_pos'], cmap='Greens')
    .background_gradient(subset=['Mean_SHAP_neg'], cmap='Reds_r')
    .format({'Mean_SHAP': '{:.6f}', 'Mean_SHAP_pos': '{:.6f}',
             'Mean_SHAP_neg': '{:.6f}', 'Std_SHAP': '{:.6f}'})
)